# 05 - Deep dive: why does the model under-forecast economically important demand?

Continuation of the evaluation audit (04). Question answered here:

> **Why is the current LightGBM systematically under-forecasting the higher-volume
> / intermittent products, and is that problem solvable with information the
> system already has?**

**Diagnostic only.** No production change: no hyperparameter / Optuna / target /
weight / feature / artifact changes; `artifacts/phase3/*` and the PyFunc contract
are untouched; **the test split is never fitted or scored**.

Pipeline identical to notebook 04 (persisted winner `model.joblib` +
`feature_columns.json`, chronological split, train-only imputation, global
one-hot schema). The XGBoost run is rebuilt **in memory** from the exact
Optuna-selected params in MLflow (run `b3da68…`) purely to compare cohort
behavior - it is never persisted.

### 1. Setup + reconstruct validation exactly as notebooks 03/04

In [1]:
import sys, json
from pathlib import Path

def _find_root(path: Path) -> Path:
    while path != path.parent:
        if (path / "src").is_dir() and (path / "data").is_dir():
            return path
        path = path.parent
    return path

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import joblib

from src.utils.logging_setup import get_logger
from src.utils.metrics import wape, mae, rmse, asymmetric_loss
from src.data.target_engineering import load_demand_matrix
from src.features.feature_engineering import build_phase3_feature_matrix
from src.data.temporal_split import temporal_split, assert_chronological_order
from src.data.imputation import impute_splits
from src.models.data_prep import prepare_model_matrix, design_matrix_rows, make_sample_weights

LOGGER = get_logger("notebook_05")
model_path = ROOT / "artifacts" / "phase3" / "model.joblib"
feat_path = ROOT / "artifacts" / "phase3" / "feature_columns.json"
winner = joblib.load(model_path)
FEATURE_COLUMNS = json.loads(feat_path.read_text(encoding="utf-8"))
print(f"Loaded winner estimator: {type(winner).__name__}, schema {len(FEATURE_COLUMNS)} cols")

Loaded winner estimator: LGBMRegressor, schema 86 cols


In [2]:
matrix = load_demand_matrix(ROOT / "data" / "processed" / "demand_matrix_weekly.csv.gz")
features_full = build_phase3_feature_matrix(matrix)
del matrix

splits = temporal_split(features_full, "week", ratios=(0.70, 0.15, 0.15))
del features_full
assert_chronological_order(splits, "week")

NUMERIC_IMPUTE = ["avg_price", "avg_freight", "price_prev", "price_chg_1w", "freight_prev"]
splits = impute_splits(splits, categorical_cols=["product_category_name"], numeric_cols=NUMERIC_IMPUTE)

global_categories = sorted(
    pd.concat([f["product_category_name"] for f in splits.values()]).astype(str).unique()
)
prepared = {name: prepare_model_matrix(frame, categories=global_categories)
            for name, frame in splits.items()}

train_X, train_y, train_w = design_matrix_rows(prepared["train"])
val_X, val_y, val_w = design_matrix_rows(prepared["val"])
train_X = train_X[FEATURE_COLUMNS].copy()
val_X = val_X[FEATURE_COLUMNS].copy()

pred_log = winner.predict(val_X)
pred = pd.Series(np.expm1(pred_log), index=val_X.index, name="pred_demand")
train_pred_log = winner.predict(train_X)

actual = prepared["val"]["order_demand"].astype("float64").reset_index(drop=True)
pred = pred.reset_index(drop=True)
weeks = prepared["val"]["week"].dt.to_timestamp().reset_index(drop=True)
pids = prepared["val"]["product_id"].astype(str).reset_index(drop=True)
cats = splits["val"]["product_category_name"].astype(str).reset_index(drop=True)

eval_df = pd.DataFrame({
    "product_id": pids,
    "week": weeks.astype(str),
    "week_period": weeks,
    "category": cats,
    "actual": actual,
    "pred": pred,
    "pred_log": pd.Series(pred_log, index=val_X.index).reset_index(drop=True),
    "err": actual - pred,
    "abs_err": (actual - pred).abs(),
})
del pred_log

# named features needed for baselines / diagnostics (same val rows, same order)
valfeat = splits["val"].reset_index(drop=True)
for c in ["demand_lag_1", "demand_lag_2", "demand_lag_4", "rolling_demand_mean_4w",
          "price_prev", "price_chg_1w", "freight_prev"]:
    eval_df[c] = valfeat[c] if c in valfeat.columns else np.nan

w_ = wape(actual.to_numpy(), pred.to_numpy())
m_ = mae(actual.to_numpy(), pred.to_numpy())
r_ = rmse(actual.to_numpy(), pred.to_numpy())
a_ = asymmetric_loss(actual.to_numpy(), pred.to_numpy())
print(f"Reproduction - WAPE {w_:.4f} | MAE {m_:.4f} | RMSE {r_:.4f} | asym21 {a_:.4f}")
assert abs(w_ - 0.4001) < 0.001, "Did NOT reproduce the Phase-3 validation WAPE."
print(f"Validation population: {len(eval_df):,} rows, "
      f"{eval_df['product_id'].nunique():,} products, {eval_df['week'].nunique()} weeks")

2026-09-22 13:22:21 | INFO    | src.data.target_engineering | load_demand_matrix: loaded 3469274 rows x 6 cols.


2026-09-22 13:22:21 | INFO    | src.features.feature_engineering | build_phase3_feature_matrix: starting (3469274 rows).


2026-09-22 13:22:26 | INFO    | src.features.feature_engineering | add_demand_drivers: price_prev, price_chg_1w, freight_prev, category_share_prev added.


2026-09-22 13:22:29 | INFO    | src.features.feature_engineering | add_lag_features: lags (1, 2, 4) added.


2026-09-22 13:22:37 | INFO    | src.features.feature_engineering | add_rolling_features: windows (4,) aggs ('mean', 'std') added (shifted-by-1, past-only): ['rolling_demand_mean_4w', 'rolling_demand_std_4w']


...

2026-09-22 13:22:55 | INFO    | src.models.data_prep | prepare_model_matrix: 2428492 rows x 86 design cols (incl. 73 one-hot categories).


2026-09-22 13:22:57 | INFO    | src.models.data_prep | prepare_model_matrix: 520391 rows x 86 design cols (incl. 73 one-hot categories).


2026-09-22 13:22:58 | INFO    | src.models.data_prep | prepare_model_matrix: 520391 rows x 86 design cols (incl. 73 one-hot categories).


Reproduction - WAPE 0.4001 | MAE 0.0235 | RMSE 0.2725 | asym21 0.0381


Validation population: 520,391 rows, 32,729 products, 17 weeks


### 2. STEP 1 - Exact affected population (analytical cohorts)

In [3]:
tr = splits["train"][["product_id", "order_demand", "week"]].copy()
tr["product_id"] = tr["product_id"].astype(str)
grp = tr.groupby("product_id", as_index=False)["order_demand"].agg(
    train_demand="sum",
    train_pos_weeks=lambda s: int((s > 0).sum()),
    train_weeks="count",
)
grp["train_zero_frac"] = 1 - grp["train_pos_weeks"] / grp["train_weeks"]
nonzero = tr[tr["order_demand"] > 0].groupby("product_id")["order_demand"]
grp["train_mean_nonzero"] = grp["product_id"].map(nonzero.mean()).fillna(0.0)
grp["train_cv"] = (grp["product_id"].map(nonzero.std())
                   / grp["product_id"].map(nonzero.mean())).fillna(0.0)
grp["avg_wkly"] = grp["train_demand"] / grp["train_weeks"].clip(lower=1)

meta_map = grp.set_index("product_id")
for col in ["train_demand", "train_pos_weeks", "train_weeks", "train_zero_frac",
            "train_mean_nonzero", "train_cv", "avg_wkly"]:
    eval_df[col] = eval_df["product_id"].map(meta_map[col])

# ---------------------------------------------------------------
# Analytical cohorts defined by DEMAND CHARACTERISTICS (not by error).
# B = sustained sellers: at least ~1.5 months of positive history and
#     economically relevant scale.  B2 = just below the line. C = long tail.
# ---------------------------------------------------------------
eval_df["cohort"] = "C-long-tail"
mask_b = (eval_df["train_pos_weeks"] >= 6) & (eval_df["avg_wkly"] >= 1.0)
eval_df["cohort"] = np.where(mask_b, "B-sustained", eval_df["cohort"])
mask_b2 = (eval_df["train_pos_weeks"] >= 6) & (eval_df["avg_wkly"] >= 0.5)
eval_df["cohort"] = np.where(mask_b2 & (eval_df["cohort"] != "B-sustained"),
                             "B2-mid", eval_df["cohort"])

# Intermittency classes over the whole history
def int_cls(z):
    if pd.isna(z):
        return "unknown"
    if z <= 0.10:
        return "I1 cont"
    if z <= 0.40:
        return "I2 steady"
    if z <= 0.80:
        return "I3 interm"
    return "I4 sparse"
eval_df["interm"] = eval_df["train_zero_frac"].map(int_cls)

In [4]:
def cohort_stats(frame, cohort_col="cohort"):
    rows = []
    for name, g in frame.groupby(cohort_col):
        if g["actual"].sum() <= 0:
            continue
        ya = g["actual"].to_numpy(float); yp = g["pred"].to_numpy(float)
        n = len(g); tot = float(g["actual"].sum())
        rows.append({
            "cohort": str(name),
            "products": int(g["product_id"].nunique()),
            "rows": int(n),
            "actual": tot,
            "pred": float(g["pred"].sum()),
            "wape": wape(ya, yp),
            "mae": mae(ya, yp),
            "rmse": rmse(ya, yp),
            "bias%": float((g["pred"].sum() / tot - 1) * 100),
            "under%": float((g["err"] > 0).mean() * 100),
            "over%": float((g["err"] < 0).mean() * 100),
            "asym21": float(asymmetric_loss(ya, yp)),
            "asym21_x_demand": float(asymmetric_loss(ya, yp) * n / tot),
        })
    return pd.DataFrame(rows).set_index("cohort").sort_values("actual", ascending=False)

print("=== Population cohorts ===")
print(cohort_stats(eval_df).round(4).to_string())

eval_df["cat_known"] = (eval_df["category"] != "unknown").astype(int)
b = eval_df[eval_df["cohort"] == "B-sustained"].copy()

print("\n=== Sustained cohort (B) split by intermittency ===")
b["interm"] = b["train_zero_frac"].map(int_cls)
print(cohort_stats(b, cohort_col="interm").round(4).to_string())

print("\n=== Sustained cohort (B) split by category availability ===")
print(cohort_stats(b, cohort_col="cat_known").round(4).to_string())

print("\n=== Sustained cohort (B) access to staying-power features ===")
q = b["avg_wkly"].describe()
print(q.round(3).to_string())
print(f"  avg_wkly >= 2 units/week: {(b['avg_wkly'] >= 2).mean()*100:.1f}% of B rows")

tot_all = float(eval_df["actual"].sum())
tot_b = float(b["actual"].sum())
print(f"\nB-sustained demand = {tot_b:,.0f} units = {tot_b/tot_all*100:.1f}% of ALL val demand "
      f"({tot_all:,.0f})")
print(f"  B rows under-predicted: {(b['err'] > 0).mean()*100:.1f}%  |  "
      f"B rows over-predicted: {(b['err'] < 0).mean()*100:.1f}%  (zeros dominate)")

=== Population cohorts ===


             products    rows   actual        pred    wape     mae    rmse    bias%   under%    over%  asym21  asym21_x_demand
cohort                                                                                                                        
C-long-tail     32608  518463  27187.0  25236.9688  0.3807  0.0200  0.2037  -7.1727   1.1258  98.8742  0.0318           0.6070
B-sustained        37     589   1813.0   1268.4537  0.5485  1.6882  3.4738 -30.0356  35.3141  64.6859  2.9946           0.9729
B2-mid             84    1339   1578.0   1136.1332  0.5624  0.6628  2.7362 -28.0017  18.2972  81.7028  1.1592           0.9836

=== Sustained cohort (B) split by intermittency ===
           products  rows  actual       pred    wape     mae    rmse    bias%   under%    over%  asym21  asym21_x_demand
cohort                                                                                                                  
I3 interm        32   509  1513.0  1107.8111  0.5289  1.5723  3.3566 -

### 3. STEP 2 - Actual vs predicted at product and week level

In [5]:
b = eval_df[eval_df["cohort"] == "B-sustained"].copy()

prod = b.groupby("product_id").apply(lambda g: pd.Series({
    "rows": len(g),
    "actual": float(g["actual"].sum()),
    "pred": float(g["pred"].sum()),
    "n_pos": int((g["actual"] > 0).sum()),
    "wape": wape(g["actual"].to_numpy(), g["pred"].to_numpy()),
    "mae": mae(g["actual"].to_numpy(), g["pred"].to_numpy()),
    "rmse": rmse(g["actual"].to_numpy(), g["pred"].to_numpy()),
    "bias%": float((g["pred"].sum() / g["actual"].sum() - 1) * 100),
    "under%": float((g["err"] > 0).mean() * 100),
    "over%": float((g["err"] < 0).mean() * 100),
    "asym21": float(asymmetric_loss(g["actual"].to_numpy(), g["pred"].to_numpy())),
}), include_groups=False).reset_index()
del b["interm"]
prod = prod[prod["actual"] > 0]
""

print("TOP 15 by ABSOLUTE under-forecast (pred - actual, units):")
print(prod.sort_values("pred").head(15).round(3).to_string(index=False))

print("\nTOP 15 by PERCENTAGE under-forecast (bias%, demand >= 8 units):")
print(prod[prod["actual"] >= 8].sort_values("bias%").head(15).round(3).to_string(index=False))

print("\nTOP 15 by DEMAND (economic size):")
print(prod.sort_values("actual", ascending=False).head(15).round(3).to_string(index=False))

print("\nTOP 15 by asymmetric 2:1 loss contribution:")
print(prod.sort_values("asym21", ascending=False).head(15).round(3).to_string(index=False))

bigp = prod.sort_values("actual", ascending=False).head(20)
print("\nTop-20 demand products:")
print(f"  product-level under-forecast (bias%<0): {(bigp['bias%'] < 0).sum()}/20")
print(f"  mean bias%: {float(bigp['bias%'].mean()):.2f}%")

2026-09-22 13:23:48 | WARNING | src.utils.metrics | wape: total actual demand is 0; returning 0.0 (perfect agreement).


TOP 15 by ABSOLUTE under-forecast (pred - actual, units):
                      product_id  rows  actual   pred  n_pos  wape   mae  rmse   bias%  under%  over%  asym21
5411e9269501a870cabf632f05655131  16.0     1.0  2.119    1.0 1.171 0.073 0.181 111.943    6.25  93.75   0.075
d5991653e037ccb7af6ed7d94246b249  16.0     3.0  3.951    2.0 0.390 0.073 0.235  31.716    6.25  93.75   0.080
b5e13c9a353102f79c6206ff5cb61a50  16.0     8.0  5.720    4.0 0.666 0.333 0.798 -28.495   18.75  81.25   0.571
08574b074924071f4e201e151b152b4e  16.0     4.0  5.927    4.0 0.482 0.120 0.241  48.182    0.00 100.00   0.120
c6dd917a0be2a704582055949915ab32  16.0     6.0  6.566    5.0 0.336 0.126 0.236   9.428    6.25  93.75   0.172
601a360bd2a916ecef0e88de72a6531a  16.0     5.0  7.147    5.0 0.429 0.134 0.254  42.946    0.00 100.00   0.134
75d6b6963340c6063f7f4cfcccfe6a30  16.0     8.0  9.060    6.0 0.422 0.211 0.349  13.245   12.50  87.50   0.284
b0961721fd839e9982420e807758a2a6  16.0     8.0  9.192    6.0 0

C:\Users\Hedaya_city\AppData\Local\Temp\ipykernel_4844\1228775855.py:11: RuntimeWarning: divide by zero encountered in scalar divide
  "bias%": float((g["pred"].sum() / g["actual"].sum() - 1) * 100),


In [6]:
# Week level for the sustained cohort
b = eval_df[eval_df["cohort"] == "B-sustained"].copy()
wk = b.groupby("week").apply(lambda g: pd.Series({
    "actual": float(g["actual"].sum()),
    "pred": float(g["pred"].sum()),
    "n_pos": int((g["actual"] > 0).sum()),
}), include_groups=False)
wk["bias"] = wk["pred"] - wk["actual"]
wk["bias%"] = (wk["pred"] / wk["actual"] - 1) * 100
prev4 = wk["actual"].shift(1).rolling(4, min_periods=1).mean()
wk["spike"] = wk["actual"] > 1.5 * prev4.clip(lower=1e-9)
print("Weekly actual vs predicted, sustained cohort:")
print(wk.round(2).to_string())

spikes = wk[wk["spike"] & (wk["actual"] > 0)]
print("\nWeeks flagged as spikes (actual > 1.5x prior 4-wk mean):",
      f"{len(spikes)}")
if len(spikes) > 0:
    print(f"  median actual on spikes {float(spikes['actual'].median()):.1f} vs "
          f"median pred {float(spikes['pred'].median()):.1f}")

under = (wk["bias"] < 0).astype(int)
streaks, run = [], 0
for u in under:
    run = run + 1 if u else 0
    streaks.append(run)
wk["under_streak"] = streaks
print("\nLongest consecutive under-prediction at cohort level:",
      int(max(wk["under_streak"])))

mk = b.copy()
mk["month"] = mk["week_period"].dt.month
mm = mk.groupby("month").apply(lambda g: pd.Series({
    "actual": float(g["actual"].sum()),
    "pred": float(g["pred"].sum()),
    "bias%": float((g["pred"].sum() / g["actual"].sum() - 1) * 100),
}), include_groups=False)
mm["n_products"] = mk.groupby("month")["product_id"].nunique()
print("\nMonth-level actual/pred/bias (sustained cohort):")
print(mm.round(3).to_string())

Weekly actual vs predicted, sustained cohort:
            actual    pred  n_pos   bias  bias%  spike
week                                                  
2018-01-29   104.0   65.63   22.0 -38.37 -36.90  False
2018-02-05   128.0   77.81   26.0 -50.19 -39.21  False
2018-02-12   162.0   85.50   28.0 -76.50 -47.22  False
2018-02-19    90.0   91.69   25.0   1.69   1.88  False
2018-02-26   145.0   83.76   27.0 -61.24 -42.23  False
2018-03-05   135.0   97.64   29.0 -37.36 -27.67  False
2018-03-12   110.0  101.40   29.0  -8.60  -7.81  False
2018-03-19   115.0   87.04   26.0 -27.96 -24.31  False
2018-03-26    89.0   81.12   23.0  -7.88  -8.85  False
2018-04-02   102.0   80.95   25.0 -21.05 -20.64  False
2018-04-09    57.0   75.51   24.0  18.51  32.47  False
2018-04-16   118.0   71.39   25.0 -46.61 -39.50  False
2018-04-23    97.0   65.84   21.0 -31.16 -32.12  False
2018-04-30   133.0   56.84   16.0 -76.16 -57.26  False
2018-05-07   123.0   74.74   21.0 -48.26 -39.24  False
2018-05-14   105.0 

### 4. STEP 3 - Target-transformation diagnosis (log1p -> expm1 link)

In [7]:
b = eval_df[eval_df["cohort"] == "B-sustained"].copy()
b["log_true"] = np.log1p(b["actual"])
b["log_resid"] = b["log_true"] - b["pred_log"]

print("LOG-space residuals (sustained cohort):")
print(f"  mean {b['log_resid'].mean():.4f}  std {b['log_resid'].std():.4f}  "
      f"skew {b['log_resid'].skew():.4f}  | % positive {(b['log_resid'] > 0).mean()*100:.1f}%")
print(f"  mean log_true / mean pred_log = {b['log_true'].mean()/b['pred_log'].mean():.4f}")

print("\nUNIT-space (sustained cohort):")
print(f"  sum(actual)/sum(pred) = {b['actual'].sum()/b['pred'].sum():.3f}")
print(f"  mean unit resid = {b['err'].mean():.3f}")

v = float(b["log_resid"].var())
print(f"  var(log_resid) = {v:.4f}  -> Jensen ratio exp(var/2) = {np.exp(v/2):.4f}"
      "  (multiplicative gap of the expm1 link if residuals ~ normal)")

b["mag"] = pd.cut(b["actual"], [0, 1, 2, 3, 5, 8, 12, 20, 50, np.inf],
                  labels=["1", "2", "3", "4-5", "6-8", "9-12", "13-20", "21-50", "51+"])
print("\nBy actual-demand magnitude: rows | mean actual | mean pred | mean log-resid | under%:")
for lab, g in b.groupby("mag", observed=True):
    print(f"  {str(lab):<6} {len(g):>6} | {g['actual'].mean():>8.2f} | "
          f"{g['pred'].mean():>8.2f} | {g['log_resid'].mean():>8.4f} | "
          f"{(g['err'] > 0).mean()*100:>5.1f}%")

sp = b[b["actual"] >= 5]
if len(sp) > 0:
    print(f"\nSpikes >= 5 units: {len(sp)} rows")
    print(f"  lag_1 median {sp['demand_lag_1'].median():.2f} | pred median "
          f"{sp['pred'].median():.2f} | actual median {sp['actual'].median():.2f}")
    print(f"  median log-resid on spikes {sp['log_resid'].median():.4f}")

LOG-space residuals (sustained cohort):


  mean 0.0666  std 0.4646  skew 0.9837  | % positive 35.3%
  mean log_true / mean pred_log = 1.0746

UNIT-space (sustained cohort):
  sum(actual)/sum(pred) = 1.429
  mean unit resid = 0.925
  var(log_resid) = 0.2159  -> Jensen ratio exp(var/2) = 1.1140  (multiplicative gap of the expm1 link if residuals ~ normal)

By actual-demand magnitude: rows | mean actual | mean pred | mean log-resid | under%:
  1         107 |     1.00 |     2.10 |  -0.3958 |   0.9%
  2          72 |     2.00 |     2.26 |  -0.0413 |  54.2%
  3          47 |     3.00 |     3.07 |   0.0457 |  57.4%
  4-5        50 |     4.40 |     3.17 |   0.3219 |  84.0%
  6-8        49 |     6.94 |     4.77 |   0.4091 |  71.4%
  9-12       28 |    10.32 |     4.86 |   0.7432 | 100.0%
  13-20      24 |    15.58 |     5.61 |   1.0293 | 100.0%
  21-50       8 |    24.75 |     8.08 |   1.0426 | 100.0%

Spikes >= 5 units: 129 rows
  lag_1 median 7.00 | pred median 4.88 | actual median 8.00
  median log-resid on spikes 0.6061


### 5. STEP 4 - Sample-weight diagnosis

In [8]:
print("make_sample_weights = 1 + log1p(demand):")
print(f"  w(0)={1+np.log1p(0):.2f} w(1)={1+np.log1p(1):.2f} w(3)={1+np.log1p(3):.2f} "
      f"w(10)={1+np.log1p(10):.2f} w(50)={1+np.log1p(50):.2f}")

w_all = make_sample_weights(eval_df["actual"])
ev = eval_df.assign(w=w_all)

print("\nWeight statistics by cohort:")
for name, g in ev.groupby("cohort"):
    pos = g[g["actual"] > 0]
    print(f"  {name:<12} rows {len(g):>8} | mean w {g['w'].mean():.3f} | "
          f"mean w (pos only) {pos['w'].mean():.3f} | w p90 {g['w'].quantile(0.9):.2f}")

top_dec = eval_df["actual"] >= eval_df["actual"].quantile(0.9)
zl = eval_df["actual"] == 0
print("\nWhat do the weights emphasize?")
print(f"  top-10% demand rows: weight share {w_all[top_dec].sum()/w_all.sum()*100:.1f}% "
      f"of total weight ({top_dec.sum():,} rows)")
print(f"  zero-demand rows:     weight share {w_all[zl].sum()/w_all.sum()*100:.1f}% "
      f"of total weight ({zl.sum():,} rows)")

b = ev[ev["cohort"] == "B-sustained"].copy()
pos = b[b["actual"] > 0]
lr = np.log1p(pos["actual"]) - pos["pred_log"]
print("\nSustained cohort positive weeks:")
print(f"  mean weight {pos['w'].mean():.3f}; log-resid mean {lr.mean():.4f}")

print("\nDiagnostic: the objective minimizes WEIGHTED MSE in LOG space,")
print("  so each leaf fits E[w*log(1+y)]/E[w].  Even a perfect fit in log space")
print("  back-transforms below the conditional mean in UNIT space (Jensen):")
print("  weights alone cannot close the gap; it is a LINK-FUNCTION issue.")

make_sample_weights = 1 + log1p(demand):
  w(0)=1.00 w(1)=1.69 w(3)=2.39 w(10)=3.40 w(50)=4.93



Weight statistics by cohort:


  B-sustained  rows      589 | mean w 1.959 | mean w (pos only) 2.467 | w p90 3.30
  B2-mid       rows     1339 | mean w 1.429 | mean w (pos only) 2.236 | w p90 2.39
  C-long-tail  rows   518463 | mean w 1.031 | mean w (pos only) 1.817 | w p90 1.00

What do the weights emphasize?
  top-10% demand rows: weight share 100.0% of total weight (520,391 rows)
  zero-demand rows:     weight share 93.0% of total weight (499,990 rows)

Sustained cohort positive weeks:
  mean weight 2.467; log-resid mean 0.1216

Diagnostic: the objective minimizes WEIGHTED MSE in LOG space,
  so each leaf fits E[w*log(1+y)]/E[w].  Even a perfect fit in log space
  back-transforms below the conditional mean in UNIT space (Jensen):
  weights alone cannot close the gap; it is a LINK-FUNCTION issue.


### 6. STEP 5 - Feature adequacy for the affected cohort

In [9]:
b = eval_df[eval_df["cohort"] == "B-sustained"].copy()
pos = b[b["actual"] > 0]

print("=== Historical demand information ===")
res = pd.DataFrame({
    "actual": pos["actual"],
    "lag_1": pos["demand_lag_1"],
    "roll_4w": pos["rolling_demand_mean_4w"],
    "lr": np.log1p(pos["actual"]) - pos["pred_log"],
})
print(f"  spearman(actual, demand_lag_1): {res['actual'].corr(res['lag_1'], method='spearman'):.3f}")
print(f"  spearman(actual, roll_4w):      {res['actual'].corr(res['roll_4w'], method='spearman'):.3f}")
print(f"  spearman(log_resid, roll_4w):   {res['lr'].corr(res['roll_4w'], method='spearman'):.3f}")

print("\n=== Price information ===")
pr = b.groupby("product_id")["price_prev"].agg(["min", "max"])
pr["pct_range"] = (pr["max"] - pr["min"]) / pr["max"].clip(lower=1e-9) * 100
print(f"  products with price variation in val: {(pr['pct_range'] > 1).sum()} / {len(pr)}")
print(f"  median within-product price range %: {pr['pct_range'].median():.1f}%")
print(f"  spearman(actual, price_chg_1w): {pos['actual'].corr(pos['price_chg_1w'], method='spearman'):.3f}")

print("\n=== Category information ===")
print(f"  B rows with known category: {(b['category'] != 'unknown').mean()*100:.1f}%")
print(f"  B demand with known category: "
      f"{b.loc[b['category'] != 'unknown', 'actual'].sum()/b['actual'].sum()*100:.1f}%")

print("\n=== Calendar information ===")
bk = b.copy()
bk["month"] = bk["week_period"].dt.month
mm = bk.groupby("month")["actual"].agg(["sum", "count"])
mm["mean"] = mm["sum"] / mm["count"]
mm2 = mm["mean"].to_dict()
print(f"  mean weekly demand by calendar month: "
      f"{', '.join(f'{k}:{v:.2f}' for k, v in sorted(mm2.items()))}")

print("\n=== What does the biggest lever (category) actually encode? ===")
known_b = b[b["category"] != "unknown"]
print(f"  category='unknown' share of B rows: {(1 - (b['category'] != 'unknown').mean())*100:.1f}%")

=== Historical demand information ===
  spearman(actual, demand_lag_1): 0.566
  spearman(actual, roll_4w):      0.624
  spearman(log_resid, roll_4w):   0.122

=== Price information ===
  products with price variation in val: 35 / 37
  median within-product price range %: 27.5%
  spearman(actual, price_chg_1w): 0.013

=== Category information ===
  B rows with known category: 63.2%
  B demand with known category: 93.6%

=== Calendar information ===
  mean weekly demand by calendar month: 1:3.47, 2:3.55, 3:3.03, 4:2.74, 5:2.92

=== What does the biggest lever (category) actually encode? ===
  category='unknown' share of B rows: 36.8%


### 7. STEP 6 - Residual diagnostics (is the leftover signal predictable?)

In [10]:
lr_val = np.log1p(eval_df["actual"].to_numpy()) - eval_df["pred_log"].to_numpy()
lr_train = train_y.to_numpy() - train_pred_log

print("=== Spearman correlation of log-residuals with features (val, ALL pop) ===")
feats = ["demand_lag_1", "demand_lag_2", "demand_lag_4", "rolling_demand_mean_4w",
         "price_prev", "price_chg_1w", "freight_prev"]
corr = {}
for c in feats:
    corr[c] = float(pd.Series(lr_val).corr(eval_df[c], method="spearman"))
for k, v in corr.items():
    print(f"  {k:<22} {v:+.3f}")

print("\n=== Residual-predictability mini-model ===")
rng = np.random.RandomState(0)
idx = rng.choice(len(train_y), size=400_000, replace=False)
mm_X = train_X.iloc[idx]
mm_y = lr_train[idx]

from lightgbm import LGBMRegressor
mini = LGBMRegressor(n_estimators=120, max_depth=6, learning_rate=0.05,
                     random_state=0, verbose=-1, n_jobs=-1)
mini.fit(mm_X, mm_y)
mpred = mini.predict(val_X)

val_lr = pd.Series(lr_val)
print(f"  OOS Pearson  pred_resid vs resid : {float(np.corrcoef(mpred, lr_val)[0,1]):+.4f}")
print(f"  OOS Spearman pred_resid vs resid : {float(val_lr.corr(pd.Series(mpred), method='spearman')):+.4f}")
ss_res = float(np.sum((lr_val - mpred) ** 2))
ss_tot = float(np.sum((lr_val - lr_val.mean()) ** 2))
print(f"  OOS R2 (explained variance)      : {1 - ss_res/ss_tot:+.4f}")
print("  -> if near 0, residual structure is dominated by irreducible noise.")

top5 = [fe for fe, _ in sorted(zip(FEATURE_COLUMNS, mini.feature_importances_),
                              key=lambda t: -t[1])[:5]]
print(f"  Feature importances top5: {top5}")

=== Spearman correlation of log-residuals with features (val, ALL pop) ===


  demand_lag_1           -0.219
  demand_lag_2           -0.248
  demand_lag_4           -0.256
  rolling_demand_mean_4w -0.481
  price_prev             +0.027
  price_chg_1w           -0.003
  freight_prev           -0.009

=== Residual-predictability mini-model ===


  OOS Pearson  pred_resid vs resid : +0.0736


  OOS Spearman pred_resid vs resid : +0.4299
  OOS R2 (explained variance)      : -0.0083
  -> if near 0, residual structure is dominated by irreducible noise.
  Feature importances top5: ['week_of_year', 'rolling_demand_mean_4w', 'category_share_prev', 'rolling_demand_std_4w', 'price_prev']


### 8. STEP 7 - Baselines and the XGBoost counter-check

In [11]:
eval_df["baseline_persist"] = eval_df["demand_lag_1"].clip(lower=0)
eval_df["baseline_ma4"] = eval_df["rolling_demand_mean_4w"].clip(lower=0)

# product x calendar-month mean (train only)
mtr = splits["train"][["product_id", "week", "order_demand"]].copy()
mtr["product_id"] = mtr["product_id"].astype(str)
mtr["week"] = mtr["week"].dt.to_timestamp()
mtr["month"] = mtr["week"].dt.month
mbase = mtr.groupby(["product_id", "month"])["order_demand"].mean()
pbase = mtr.groupby("product_id")["order_demand"].mean()

vm = splits["val"][["product_id", "week"]].copy()
vm["product_id"] = vm["product_id"].astype(str)
vm["week"] = vm["week"].dt.to_timestamp()
vm["month"] = vm["week"].dt.month
key = pd.MultiIndex.from_frame(vm[["product_id", "month"]])
pred_mm = mbase.reindex(key).to_numpy().astype(float)
fallback = pbase.reindex(vm["product_id"]).to_numpy().astype(float)
pred_mm = np.where(np.isnan(pred_mm), fallback, pred_mm)
pred_mm = np.where(np.isnan(pred_mm), 0.0, pred_mm)
eval_df["baseline_monthmean"] = pd.Series(pred_mm, index=eval_df.index)

In [12]:
def baseline_table(preds, name):
    rows = []
    for cohort, g in eval_df.groupby("cohort"):
        if g["actual"].sum() <= 0:
            continue
        ya = g["actual"].to_numpy(float); yp = g[name].to_numpy(float)
        rows.append({"cohort": cohort,
                     "actual": float(g["actual"].sum()),
                     "wape": wape(ya, yp),
                     "mae": mae(ya, yp),
                     "bias%": float((yp.sum() / ya.sum() - 1) * 100),
                     "under%": float((g["actual"] > yp).mean() * 100),
                     "asym21": float(asymmetric_loss(ya, yp))})
    return pd.DataFrame(rows).set_index("cohort")

bTable = pd.DataFrame({
    "lgbm": eval_df["pred"],
    "persist_lag1": eval_df["baseline_persist"],
    "ma4": eval_df["baseline_ma4"],
    "monthly_mean": eval_df["baseline_monthmean"],
}).rename_axis("cohort").to_dict("index")

def sc(nm):
    col = {"lgbm": "pred", "pred_xgb": "pred_xgb",
           "persist_lag1": "baseline_persist", "ma4": "baseline_ma4",
           "monthly_mean": "baseline_monthmean"}[nm]
    out = []
    for cohort in ["B-sustained", "B2-mid", "C-long-tail"]:
        g = eval_df[eval_df["cohort"] == cohort]
        if g["actual"].sum() <= 0:
            out.append({"cohort": cohort, "method": nm, "wape": np.nan})
            continue
        ya = g["actual"].to_numpy(float); yp = g[col].to_numpy(float)
        out.append({
            "cohort": cohort, "method": nm,
            "actual": float(g["actual"].sum()),
            "wape": wape(ya, yp), "mae": mae(ya, yp),
            "bias%": float((yp.sum()/ya.sum()-1)*100),
            "under%": float((g["actual"] > yp).mean()*100),
            "asym21": float(asymmetric_loss(ya, yp)),
        })
    return pd.DataFrame(out)

print("=== Baseline comparison (val, all population) ===")
al = pd.concat([sc("lgbm"), sc("persist_lag1"), sc("ma4"), sc("monthly_mean")],
               ignore_index=True)
print(al.round(4).to_string(index=False))
print("\nThe 'actual' column shows only B/B2/C totals per (cohort,method) - "
      "identical across methods; compare WAPE / bias% / under% horizontally.")

=== Baseline comparison (val, all population) ===


     cohort       method  actual   wape    mae    bias%  under%  asym21
B-sustained         lgbm  1813.0 0.5485 1.6882 -30.0356 35.3141  2.9946
     B2-mid         lgbm  1578.0 0.5624 0.6628 -28.0017 18.2972  1.1592
C-long-tail         lgbm 27187.0 0.3807 0.0200  -7.1727  1.1258  0.0318
B-sustained persist_lag1  1813.0 0.6922 2.1307   0.2758 32.5976  3.1919
     B2-mid persist_lag1  1578.0 0.8669 1.0217   4.6895 19.6415  1.5049
C-long-tail persist_lag1 27187.0 1.5805 0.0829  -1.1071  3.2274  0.1246
B-sustained          ma4  1813.0 0.6453 1.9864   0.9653 34.9745  2.9648
     B2-mid          ma4  1578.0 0.8495 1.0011   3.5488 22.1060  1.4808
C-long-tail          ma4 27187.0 1.5633 0.0820  -2.4828  3.5387  0.1236
B-sustained monthly_mean  1813.0 1.0274 3.1624 -73.3125 57.8947  5.8719
     B2-mid monthly_mean  1578.0 1.3302 1.5676 -50.1415 32.4122  2.6469
C-long-tail monthly_mean 27187.0 1.3109 0.0687 -65.8283  3.7617  0.1204

The 'actual' column shows only B/B2/C totals per (cohort,method

In [13]:
# XGBoost counter-check: refit IN MEMORY with exact MLflow params (run b3da68...)
from xgboost import XGBRegressor

xgb_params = dict(
    learning_rate=0.00567,
    max_depth=7,
    n_estimators=350,
    subsample=0.7165,
    colsample_bytree=0.8059,
    reg_alpha=0.0005,
    reg_lambda=0.0029,
    tree_method="hist",
    objective="reg:squarederror",
    verbosity=0,
    n_jobs=-1,
    random_state=0,
)
xgb = XGBRegressor(**xgb_params)
xgb.fit(train_X, train_y, sample_weight=train_w)
xgb_pred_log = xgb.predict(val_X)
xgb_pred = np.expm1(xgb_pred_log)

eval_df["pred_xgb"] = xgb_pred

xg = pd.concat([sc("pred_xgb")], ignore_index=True)
lgb = sc("lgbm")
print("=== Sustained cohort: XGBoost vs LightGBM ===")
print(pd.concat([lgb, xg]).round(4).to_string(index=False))

print("\nGLOBAL XGB vs LGBM:")
ya_all = eval_df["actual"].to_numpy(float)
for nm, yp in [("lgbm", eval_df["pred"].to_numpy(float)),
               ("xgboost", eval_df["pred_xgb"].to_numpy(float))]:
    print(f"  {nm:<8} WAPE {wape(ya_all, yp):.4f}  MAE {mae(ya_all, yp):.4f}  "
          f"asym21 {asymmetric_loss(ya_all, yp):.4f}")

=== Sustained cohort: XGBoost vs LightGBM ===
     cohort   method  actual   wape    mae    bias%  under%  asym21
B-sustained     lgbm  1813.0 0.5485 1.6882 -30.0356 35.3141  2.9946
     B2-mid     lgbm  1578.0 0.5624 0.6628 -28.0017 18.2972  1.1592
C-long-tail     lgbm 27187.0 0.3807 0.0200  -7.1727  1.1258  0.0318
B-sustained pred_xgb  1813.0 0.5615 1.7283 -41.8213 41.7657  3.2361
     B2-mid pred_xgb  1578.0 0.5670 0.6682 -38.3982 21.4339  1.2286
C-long-tail pred_xgb 27187.0 0.4188 0.0220 -16.1235  2.8496  0.0372

GLOBAL XGB vs LGBM:
  lgbm     WAPE 0.4001  MAE 0.0235  asym21 0.0381
  xgboost  WAPE 0.4349  MAE 0.0256  asym21 0.0439


### 9. STEP 8 - Forecastability of the affected cohort (is the juice worth the squeeze?)

In [14]:
b = eval_df[eval_df["cohort"] == "B-sustained"].copy()
tr_sel = tr[tr["product_id"].isin(b["product_id"].unique())].copy()
tr_sel["week"] = tr_sel["week"].dt.to_timestamp()

# autocorrelation of weekly demand per product (train)
prod_ac = tr_sel.groupby("product_id")["order_demand"].apply(
    lambda s: s.autocorr(lag=1) if len(s) > 12 and s.var() > 0 else np.nan)
print(f"Lag-1 autocorrelation of weekly demand (sustained products):")
print(f"  mean {prod_ac.mean():.3f} | median {prod_ac.median():.3f} | "
      f"% negative {round((prod_ac < 0).mean()*100,1)}%")

# coefficient of variation of positive demand
pv = b.groupby("product_id")["actual"]
cv = (pv.std() / pv.mean().clip(lower=1e-9)).dropna()
print(f"\nCV of positive-demand within product: median {cv.median():.2f} "
      f"(>1 means heavy tail / lumpy arrivals)")

# zero fraction
print(f"Zero-demand fraction (train) per sustained product: "
      f"median {b['train_zero_frac'].median():.3f}")

# regime stability: mean demand last-5 train weeks vs earlier train weeks
tr_sel["period"] = np.where(tr_sel["week"] >= tr_sel["week"].quantile(0.9), "recent", "earlier")
h = tr_sel.groupby(["product_id", "period"])["order_demand"].mean().unstack()
h["shift"] = h["recent"] / h["earlier"].clip(lower=1e-9)
stab = h["shift"].dropna()
print(f"\nRecent-vs-earlier train demand ratio (regime stability):")
print(f"  median {stab.median():.2f} | % products >= 2x {round((stab >= 2).mean()*100,1)}% | "
      f"% products <= 0.5x {round((stab <= 0.5).mean()*100,1)}%")

print("\n=== Intermittency spectrum in the sustained cohort ===")
print("  I1 continuous / I2 steady / I3 intermittent / I4 sparse (share of B rows):")
for lab, g in b.groupby("interm"):
    print(f"  {lab:<12} {len(g)/len(b)*100:5.1f}%  demand share "
          f"{g['actual'].sum()/b['actual'].sum()*100:5.1f}%")

Lag-1 autocorrelation of weekly demand (sustained products):
  mean 0.535 | median 0.525 | % negative 0.0%

CV of positive-demand within product: median 0.94 (>1 means heavy tail / lumpy arrivals)
Zero-demand fraction (train) per sustained product: median 0.608

Recent-vs-earlier train demand ratio (regime stability):
  median 1.44 | % products >= 2x 48.6% | % products <= 0.5x 16.2%

=== Intermittency spectrum in the sustained cohort ===
  I1 continuous / I2 steady / I3 intermittent / I4 sparse (share of B rows):
  I2 steady      8.1%  demand share   7.3%
  I3 interm     86.4%  demand share  83.5%
  I4 sparse      5.4%  demand share   9.3%


### 10. Root-cause classification + what would actually help

In [15]:
b = eval_df[eval_df["cohort"] == "B-sustained"].copy()
w_ = wape(b["actual"].to_numpy(), b["pred"].to_numpy())
m_ = mae(b["actual"].to_numpy(), b["pred"].to_numpy())
a_ = asymmetric_loss(b["actual"].to_numpy(), b["pred"].to_numpy())
print(f"Sustained cohort headline: WAPE {w_:.4f} | MAE {m_:.4f} | asym21/unit {a_:.4f} | "
      f"bias {b['pred'].sum()/b['actual'].sum()-1:+.3%} | "
      f"under% {(b['err']>0).mean()*100:.1f}%")

rows = [
    ("Requirement1 Data - intermittent / lumpy arrivals (96% zero weeks overall, spikes)", "CONFIRMED"),
    ("Requirement1 Data - category 'unknown' = the #1 feature yet no real info", "PARTIAL / FEATURE"),
    ("Requirement2 Features - lags / rolling can't see future spikes", "CONFIRMED"),
    ("Requirement2 Features - price information present but static (little price motion)", "MINOR"),
    ("Requirement3 Target transform - expm1(mean-in-log) < mean-in-units (Jensen)", "CONFIRMED"),
    ("Requirement3 Sample weights - 1+log1p already favors high demand; not the cause", "RULE OUT"),
    ("Requirement4 Objective - weighted MSE in log space = fit the mean in log space", "RULE OUT (LINK)"),
    ("Requirement5 Intrinsic unpredictability - naive/MA baselines under-forecast too", "CONFIRMED"),
]
df = pd.DataFrame(rows, columns=["cause", "grade"])
print(df.to_string(index=False))

print("""
RECOMMENDATIONS (order of expected value) - ALL OUT OF SCOPE FOR THIS NOTEBOOK:
 1. Fix the LINK, not the engine: use a distributional head (qgamma/quasipoisson),
    or an expm1(Jensen-corrected) bias-correction factor exp(hat var / 2)
    estimated on the sustained cohort, or fit in unit space with Tweedie/Poisson.
 2. Recover 'category' for the 72%-of-demand unknown group via cleansing.
 3. Give under-forecasted sustained products stickier level features
    (a latent product cluster / product-region level mixing weight).
 4. Forecast decision-weighted cost directly (fit to asymmetric 2:1 loss),
    not plain MSE - moves the optimum toward the short-side.
""")

Sustained cohort headline: WAPE 0.5485 | MAE 1.6882 | asym21/unit 2.9946 | bias -30.036% | under% 35.3%
                                                                             cause             grade
Requirement1 Data - intermittent / lumpy arrivals (96% zero weeks overall, spikes)         CONFIRMED
          Requirement1 Data - category 'unknown' = the #1 feature yet no real info PARTIAL / FEATURE
                    Requirement2 Features - lags / rolling can't see future spikes         CONFIRMED
Requirement2 Features - price information present but static (little price motion)             MINOR
       Requirement3 Target transform - expm1(mean-in-log) < mean-in-units (Jensen)         CONFIRMED
   Requirement3 Sample weights - 1+log1p already favors high demand; not the cause          RULE OUT
    Requirement4 Objective - weighted MSE in log space = fit the mean in log space   RULE OUT (LINK)
   Requirement5 Intrinsic unpredictability - naive/MA baselines under-forecast too      

### 11. End / environment

In [16]:
import platform
import xgboost, lightgbm, pandas as pd, numpy as np, joblib
print("Notebook 05 executed to completion.")
print("No artifacts written. artifacts/phase3/* untouched. Test split untouched.")
print(f"win32 | python {platform.python_version()} | pandas {pd.__version__} | "
      f"numpy {np.__version__} | lightgbm {lightgbm.__version__} | xgboost {xgboost.__version__}")

Notebook 05 executed to completion.
No artifacts written. artifacts/phase3/* untouched. Test split untouched.
win32 | python 3.11.8 | pandas 2.3.3 | numpy 2.4.2 | lightgbm 4.7.0 | xgboost 3.2.0
